In [1]:
# HISTORICAL EVENTS VS TEXT EMOTIONS - DETAILED CORRELATION ANALYSIS
# Compare all 11 emotions individually between text and historical events

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import os
from scipy import stats

warnings.filterwarnings("ignore")

EMOTION_LABELS = ["anger", "contempt", "disgust", "fear", "frustration", 
                "gratitude", "joy", "love", "neutral", "sadness", "surprise"]

print("="*70)
print("CORRELATION ANALYSIS: TEXT EMOTIONS vs HISTORICAL EVENTS")
print("="*70)

CORRELATION ANALYSIS: TEXT EMOTIONS vs HISTORICAL EVENTS


In [2]:
# Load all datasets
text_df = pd.read_csv("emotion_by_decade.csv")
hist_english = pd.read_csv("historical_events_english.csv")
hist_hindi = pd.read_csv("historical_events_hindi.csv")
hist_tamil = pd.read_csv("historical_events_tamil.csv")

print(f"Text data: {len(text_df)} rows")
print(f"English hist: {len(hist_english)} rows")
print(f"Hindi hist: {len(hist_hindi)} rows")
print(f"Tamil hist: {len(hist_tamil)} rows")

# Show available decades by language
for lang in ['English', 'Hindi', 'Tamil']:
    decades = sorted(text_df[text_df['language']==lang]['decade'].unique())
    print(f"{lang} text decades: {list(decades)}")

Text data: 48 rows
English hist: 16 rows
Hindi hist: 17 rows
Tamil hist: 17 rows
English text decades: [np.int64(1810), np.int64(1820), np.int64(1830), np.int64(1840), np.int64(1850), np.int64(1860), np.int64(1870), np.int64(1880), np.int64(1890), np.int64(1900), np.int64(1910), np.int64(1920), np.int64(1930), np.int64(1940), np.int64(1950), np.int64(1960)]
Hindi text decades: [np.int64(1810), np.int64(1820), np.int64(1830), np.int64(1840), np.int64(1850), np.int64(1860), np.int64(1870), np.int64(1880), np.int64(1890), np.int64(1900), np.int64(1910), np.int64(1920), np.int64(1930), np.int64(1940), np.int64(1950), np.int64(1960), np.int64(1970), np.int64(1980)]
Tamil text decades: [np.int64(1810), np.int64(1840), np.int64(1860), np.int64(1880), np.int64(1890), np.int64(1900), np.int64(1910), np.int64(1920), np.int64(1930), np.int64(1940), np.int64(1950), np.int64(1960), np.int64(1970), np.int64(1980)]


In [3]:
# Merge text and historical data
def merge_by_decade(text_lang_df, hist_df, language):
    hist_df = hist_df.copy()
    hist_df['decade'] = hist_df['decade'].astype(int)
    merged = pd.merge(
        text_lang_df, hist_df, 
        on='decade', how='inner',
        suffixes=('', '_hist')
    )
    if len(merged) > 0:
        merged['language'] = language
    return merged

# Get text data for each language
text_english = text_df[text_df['language']=='English'].copy()
text_hindi = text_df[text_df['language']=='Hindi'].copy()
text_tamil = text_df[text_df['language']=='Tamil'].copy()

# Merge
merged_english = merge_by_decade(text_english, hist_english, 'English')
merged_hindi = merge_by_decade(text_hindi, hist_hindi, 'Hindi')
merged_tamil = merge_by_decade(text_tamil, hist_tamil, 'Tamil')

all_merged = pd.concat([merged_english, merged_hindi, merged_tamil], ignore_index=True)
all_merged = all_merged.dropna(subset=['decade'])
all_merged['decade'] = all_merged['decade'].astype(int)

print(f"\nMerged data: {len(all_merged)} rows")
for lang in ['English', 'Hindi', 'Tamil']:
    count = len(all_merged[all_merged['language']==lang])
    print(f"  {lang}: {count} decades matched")


Merged data: 44 rows
  English: 16 decades matched
  Hindi: 16 decades matched
  Tamil: 12 decades matched


In [15]:
# SCATTER PLOTS: Each emotion individually
def plot_emotion_scatters(merged_df, title):
    if merged_df is None or len(merged_df) < 3:
        print(f"{title}: Not enough data")
        return
    
    # Create 11 scatter plots (4 rows x 3 cols)
    fig = make_subplots(rows=4, cols=3, subplot_titles=EMOTION_LABELS)
    
    row, col = 1, 1
    for emotion in EMOTION_LABELS:
        text_vals = merged_df[emotion].values
        hist_vals = merged_df[f'{emotion}_hist'].values
        decades = merged_df['decade'].astype(str).values
        
        fig.add_trace(go.Scatter(
            x=hist_vals, y=text_vals,
            mode='markers+text',
            text=decades,
            textposition='top center',
            marker=dict(size=12, color='blue'),
            showlegend=False
        ), row=row, col=col)
        
        # Add y=x reference line
        fig.add_trace(go.Scatter(
            x=[0, 1], y=[0, 1],
            mode='lines', line=dict(dash='dash', color='gray'),
            showlegend=False
        ), row=row, col=col)
        
        col += 1
        if col > 3:
            col = 1
            row += 1
    
    fig.update_layout(
        title=f'{title}: Text vs Historical - Each Emotion',
        height=900, template='plotly_white'
    )
    fig.update_xaxes(title_text='Historical', range=[0, 1])
    fig.update_yaxes(title_text='Text', range=[0, 1])
    fig.show()

plot_emotion_scatters(merged_english, 'English')


In [13]:
plot_emotion_scatters(merged_hindi, 'Hindi')


In [14]:
plot_emotion_scatters(merged_tamil, 'Tamil')

In [5]:
# CALCULATE CORRELATION FOR EACH EMOTION
def calculate_correlations(merged_df, title):
    if merged_df is None or len(merged_df) < 3:
        return None
    
    results = []
    for emotion in EMOTION_LABELS:
        text_vals = merged_df[emotion].values
        hist_vals = merged_df[f'{emotion}_hist'].values
        
        # Pearson correlation
        r, p = stats.pearsonr(text_vals, hist_vals)
        results.append({
            'emotion': emotion,
            'pearson_r': r,
            'pearson_p': p
        })
    
    return pd.DataFrame(results)

print("="*70)
print("CORRELATION COEFFICIENTS BY EMOTION")
print("="*70)

for lang, merged_df in [('English', merged_english), ('Hindi', merged_hindi), ('Tamil', merged_tamil)]:
    corr_df = calculate_correlations(merged_df, lang)
    if corr_df is not None:
        print(f"\n--- {lang.upper()} ---")
        # Sort by absolute correlation
        corr_df['abs_r'] = corr_df['pearson_r'].abs()
        corr_df = corr_df.sort_values('abs_r', ascending=False)
        
        for _, row in corr_df.iterrows():
            sig = "***" if row['pearson_p'] < 0.05 else "**" if row['pearson_p'] < 0.10 else ""
            print(f"  {row['emotion']:12s}: r = {row['pearson_r']:+.3f} (p = {row['pearson_p']:.3f}) {sig}")

CORRELATION COEFFICIENTS BY EMOTION

--- ENGLISH ---
  contempt    : r = -0.491 (p = 0.053) **
  anger       : r = -0.354 (p = 0.178) 
  surprise    : r = -0.286 (p = 0.284) 
  frustration : r = -0.255 (p = 0.340) 
  fear        : r = -0.184 (p = 0.495) 
  sadness     : r = -0.163 (p = 0.546) 
  joy         : r = +0.150 (p = 0.580) 
  neutral     : r = -0.103 (p = 0.703) 
  love        : r = -0.085 (p = 0.753) 
  disgust     : r = -0.079 (p = 0.772) 
  gratitude   : r = -0.004 (p = 0.989) 

--- HINDI ---
  love        : r = +0.527 (p = 0.036) ***
  joy         : r = +0.469 (p = 0.067) **
  sadness     : r = -0.368 (p = 0.161) 
  disgust     : r = -0.292 (p = 0.273) 
  neutral     : r = -0.239 (p = 0.374) 
  fear        : r = -0.199 (p = 0.460) 
  frustration : r = -0.172 (p = 0.524) 
  contempt    : r = -0.080 (p = 0.770) 
  anger       : r = -0.051 (p = 0.851) 
  gratitude   : r = +0.024 (p = 0.930) 
  surprise    : r = +0.021 (p = 0.939) 

--- TAMIL ---
  joy         : r = +0.630 (p 

In [6]:
# CORRELATION HEATMAP: Shows which emotions correlate best
fig = go.Figure()

all_data = []
emotions_list = []
languages_list = []

for lang, merged_df in [('English', merged_english), ('Hindi', merged_hindi), ('Tamil', merged_tamil)]:
    if merged_df is not None and len(merged_df) >= 3:
        row_corrs = []
        for emotion in EMOTION_LABELS:
            r, _ = stats.pearsonr(merged_df[emotion].values, merged_df[f'{emotion}_hist'].values)
            row_corrs.append(r)
        all_data.append(row_corrs)
        languages_list.append(lang)

fig.add_trace(go.Heatmap(
    z=all_data,
    x=EMOTION_LABELS,
    y=languages_list,
    colorscale='RdBu',
    zmid=0,
    colorbar=dict(title='Pearson r')
))

fig.update_layout(
    title='CORRELATION HEATMAP: Which Emotions Correlate Best?',
    height=350, template='plotly_white',
    xaxis_title='Emotion', yaxis_title='Language'
)
fig.show()

print("\nInterpretation:")
print("  +1.0 = Perfect positive correlation")
print("  -1.0 = Perfect negative correlation")
print("  0.0 = No correlation")
print("  Green = Positive correlation (text follows historical)")
print("  Red = Negative correlation (text opposes historical)")


Interpretation:
  +1.0 = Perfect positive correlation
  -1.0 = Perfect negative correlation
  0.0 = No correlation
  Green = Positive correlation (text follows historical)
  Red = Negative correlation (text opposes historical)


In [7]:
# FIND BEST AND WORST CORRELATIONS
print("="*70)
print("BEST AND WORST CORRELATIONS")
print("="*70)

all_corrs = []

for lang, merged_df in [('English', merged_english), ('Hindi', merged_hindi), ('Tamil', merged_tamil)]:
    if merged_df is not None and len(merged_df) >= 3:
        for emotion in EMOTION_LABELS:
            r, p = stats.pearsonr(merged_df[emotion].values, merged_df[f'{emotion}_hist'].values)
            all_corrs.append({'language': lang, 'emotion': emotion, 'r': r, 'p': p})

corr_all = pd.DataFrame(all_corrs)
corr_all['abs_r'] = corr_all['r'].abs()

# Top 5 positive correlations
print("\nTOP 5 POSITIVE CORRELATIONS:")
top_pos = corr_all[corr_all['r'] > 0].nlargest(5, 'r')
for _, row in top_pos.iterrows():
    print(f"  {row['language']:7s} {row['emotion']:12s}: r = {row['r']:+.3f}")

# Top 5 negative correlations
print("\nTOP 5 NEGATIVE CORRELATIONS:")
top_neg = corr_all[corr_all['r'] < 0].nsmallest(5, 'r')
for _, row in top_neg.iterrows():
    print(f"  {row['language']:7s} {row['emotion']:12s}: r = {row['r']:+.3f}")

# Top 5 absolute correlations (strongest relationships)
print("\nTOP 5 STRONGEST RELATIONSHIPS (regardless of direction):")
top_abs = corr_all.nlargest(5, 'abs_r')
for _, row in top_abs.iterrows():
    print(f"  {row['language']:7s} {row['emotion']:12s}: r = {row['r']:+.3f}")

BEST AND WORST CORRELATIONS

TOP 5 POSITIVE CORRELATIONS:
  Tamil   joy         : r = +0.630
  Hindi   love        : r = +0.527
  Tamil   surprise    : r = +0.525
  Hindi   joy         : r = +0.469
  Tamil   gratitude   : r = +0.441

TOP 5 NEGATIVE CORRELATIONS:
  English contempt    : r = -0.491
  Hindi   sadness     : r = -0.368
  English anger       : r = -0.354
  Hindi   disgust     : r = -0.292
  English surprise    : r = -0.286

TOP 5 STRONGEST RELATIONSHIPS (regardless of direction):
  Tamil   joy         : r = +0.630
  Hindi   love        : r = +0.527
  Tamil   surprise    : r = +0.525
  English contempt    : r = -0.491
  Hindi   joy         : r = +0.469


In [8]:
# HEATMAP: How much does text deviate from historical?
def plot_deviation_heatmap(merged_df, title):
    if merged_df is None or len(merged_df) == 0:
        return
    
    decades = merged_df['decade'].astype(str).tolist()
    deviations = []
    for emotion in EMOTION_LABELS:
        dev = merged_df[emotion].values - merged_df[f'{emotion}_hist'].values
        deviations.append(dev.tolist())
    
    fig = go.Figure(data=go.Heatmap(
        z=deviations,
        x=decades,
        y=EMOTION_LABELS,
        colorscale='RdBu_r',
        zmid=0,
        colorbar=dict(title='Deviation')
    ))
    
    fig.update_layout(
        title=f'{title}: Text - Historical Deviation',
        height=400, template='plotly_white',
        xaxis_title='Decade', yaxis_title='Emotion'
    )
    fig.show()

plot_deviation_heatmap(merged_english, 'English')
plot_deviation_heatmap(merged_hindi, 'Hindi')
plot_deviation_heatmap(merged_tamil, 'Tamil')

In [9]:
# NET SENTIMENT SCATTER: All emotions combined
fig = make_subplots(rows=1, cols=3, subplot_titles=('English', 'Hindi', 'Tamil'))

POSITIVE = ['gratitude', 'joy', 'love']
NEGATIVE = ['anger', 'contempt', 'disgust', 'fear', 'frustration', 'sadness']

col = 1
for lang, merged_df in [('English', merged_english), ('Hindi', merged_hindi), ('Tamil', merged_tamil)]:
    if merged_df is not None and len(merged_df) > 0:
        # Calculate net
        text_pos = merged_df[POSITIVE].mean(axis=1)
        text_neg = merged_df[NEGATIVE].mean(axis=1)
        hist_pos = merged_df[[f'{e}_hist' for e in POSITIVE]].mean(axis=1)
        hist_neg = merged_df[[f'{e}_hist' for e in NEGATIVE]].mean(axis=1)
        
        text_net = text_pos - text_neg
        hist_net = hist_pos - hist_neg
        
        fig.add_trace(go.Scatter(
            x=hist_net, y=text_net,
            mode='markers+text',
            text=merged_df['decade'].astype(str),
            textposition='top center',
            marker=dict(size=12)
        ), row=1, col=col)
    col += 1

fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.add_vline(x=0, line_dash='dash', line_color='gray')

fig.update_layout(
    title='NET SENTIMENT (Positive - Negative): Text vs Historical',
    height=400, template='plotly_white'
)
fig.show()

In [10]:
# Export complete comparison
export_cols = ['decade', 'language', 'event'] + EMOTION_LABELS + [f'{e}_hist' for e in EMOTION_LABELS]

export_df = all_merged[export_cols].copy()
export_df = export_df.sort_values(['language', 'decade'])

export_df.to_csv('sentiment_correlation.csv', index=False)

print(f"Exported to sentiment_correlation.csv")
print(f"Total rows: {len(export_df)}")
for lang in ['English', 'Hindi', 'Tamil']:
    count = len(export_df[export_df['language']==lang])
    print(f"  {lang}: {count}")

Exported to sentiment_correlation.csv
Total rows: 44
  English: 16
  Hindi: 16
  Tamil: 12
